# Perbandingan run baseline
Jalankan dari root repositori (kernel notebook harus memakai `.venv` yang sama dengan proyek). Ubah `SPLIT` ke `test` hanya untuk laporan akhir.

In [ ]:
import os, sys
from pathlib import Path

# pastikan root repositori ada di sys.path dan jadi direktori kerja
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import polars as pl
import matplotlib.pyplot as plt
from src.evaluation.compare_runs import load_runs, summary_table, naive_reference, comparability_warnings
from src.evaluation import plots

%matplotlib inline
EXPS, SPLIT = "exps", "val"

## 1. Muat run dan tabel ringkasan
Hanya folder yang punya `eval_<split>.json` yang ikut. Perhatikan peringatan: run dengan `n_samples` berbeda tidak sebanding untuk coverage, lebar interval, dan PIT.

In [ ]:
runs = load_runs(EXPS, SPLIT)
print(f"{len(runs)} run:", list(runs))
for w in comparability_warnings(runs):
    print("PERINGATAN:", w)

df = summary_table(runs, sort_by="crps")
with pl.Config(tbl_rows=100, tbl_cols=-1, tbl_width_chars=250, float_precision=4):
    print(df)

Kolom penting: `mae_vs_naive_pct` dan `mse_vs_naive_pct` (negatif = lebih baik dari seasonal naive), `cov50/80/95` (bandingkan dengan nominal, ingat bias jumlah sampel), `pit_skew` (positif = nilai nyata sering di atas sampel prediksi, yaitu bias ke bawah), dan `pit_tails` (makin besar makin overconfident).

## 2. Metrik utama per run

In [ ]:
plots.plot_metric_bars(df, naive=naive_reference(runs))
plt.show()

## 3. MAE per langkah horizon

In [ ]:
plots.plot_per_horizon(runs, 'mae_median')
plt.show()

## 4. MAE per kanal

In [ ]:
plots.plot_per_channel(runs, 'mae_median')
plt.show()

## 5. Kalibrasi interval

In [ ]:
plots.plot_calibration(runs)
plt.show()

## 6. Histogram PIT

In [ ]:
plots.plot_pit(runs)
plt.show()

## 7. Kurva training

In [ ]:
plots.plot_training_curves(runs)
plt.show()

## 8. Simpan tabel dan gambar
Keluaran ke `exps/_compare/`. Kirim `summary_<split>.csv` dan gambarnya untuk dianalisis.

In [ ]:
paths = plots.save_all(runs, df, Path(EXPS) / "_compare", split=SPLIT)
for p in paths:
    print(p)